# 08 — Scores, thresholds, ROC & AUC

*Module 00 · Getting Started — notebook 8 of 11*

Notebook 07 judged the model at a single cutoff. But a classifier has a dial behind that cutoff — a score — and turning it trades precision for recall. This notebook finds the dial, and the tools to read it.

**Prerequisites:** 05 (the centroid score idea), 07 (precision and recall).

**What you'll be able to do**
- Turn a classifier's output into a **score**, not only a label.
- See how moving the **decision threshold** trades precision against recall.
- Read a **ROC curve** and its **AUC**, and a **precision-recall curve**.
- Tell a *threshold-fixed* metric (accuracy, precision, recall) from a *threshold-swept* one (AUC).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestCentroid

from ml_course import colors, datasets, viz

np.random.seed(0)
viz.use_course_style()

X, y = datasets.penguins_xy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)
y_true = (y_test == 'Gentoo').to_numpy().astype(int)  # positive class = Gentoo


def gentoo_score(X_train_f, X_test_f):
    '''Signed squared-distance score: high when a penguin is closer to the Gentoo centroid.'''
    centroids = X_train_f.groupby(y_train).mean()
    mu_adelie = centroids.loc['Adelie'].to_numpy()
    mu_gentoo = centroids.loc['Gentoo'].to_numpy()
    pts = X_test_f.to_numpy()
    d_adelie = ((pts - mu_adelie) ** 2).sum(axis=1)
    d_gentoo = ((pts - mu_gentoo) ** 2).sum(axis=1)
    return d_adelie - d_gentoo


bill = ['bill_length_mm']
scores = gentoo_score(X_train[bill], X_test[bill])

## Where we are

Notebook 07 gave us precision and recall — but at one fixed decision: call a penguin Gentoo when the model says Gentoo. That hides a choice. Underneath the label is a continuous **score** of how Gentoo-like a penguin is, and *where we put the cutoff on that score* is ours to set. Move it, and precision and recall move together.

## From a label to a score

Our nearest-centroid classifier already computes something continuous: the distance to each centroid. Turn the two distances into one number — the **signed squared-distance**, with Gentoo as the positive class:

$$ s(x) = d(x, \mu_{\text{Adelie}})^2 - d(x, \mu_{\text{Gentoo}})^2. $$

It is large and positive when a penguin sits far from the Adélie centroid and close to the Gentoo one — that is, when it looks Gentoo. Because nearest-centroid here measures plain (Euclidean) distance to the two class means, predicting Gentoo exactly when $s > 0$ *is* the nearest-centroid rule from notebook 05; we are only exposing the number it was thresholding all along.

In [ ]:
nearest_centroid = NearestCentroid().fit(X_train[bill], y_train)
nc_pred = nearest_centroid.predict(X_test[bill])

print('a few test penguins, sorted by score (most Adelie-like first):')
order = np.argsort(scores)
for i in list(order[:3]) + list(order[-3:]):
    print(f'  score {scores[i]:+7.1f}   true species: {y_test.iloc[i]}')

agree = bool((np.where(scores > 0, 'Gentoo', 'Adelie') == nc_pred).all())
print(f'\n(score > 0) reproduces the nearest-centroid label: {agree}')

### Read the output

Sorted by score, the most Adélie-like penguins sit at the bottom (large negative $s$) and the most Gentoo-like at the top. The label we used before was only the *sign* of this score — and indeed $s > 0$ reproduces the nearest-centroid prediction exactly. The score carries more than the label: it says *how* Gentoo-like a penguin is, not only *whether*.

## The threshold dial

Nothing forces the cutoff to be zero. **Raise** it and we call a penguin Gentoo only when the score is high — fewer false alarms (precision up) but more misses (recall down). **Lower** it and we catch more Gentoos (recall up) at the cost of more false alarms (precision down). The threshold is a dial between the two kinds of error.

In [ ]:
viz.plot_score_threshold(scores, y_true, threshold=0.0, class_names=('Adelie', 'Gentoo'))
plt.show()

print('threshold    precision   recall')
for t in [-50, 0, 50]:
    pred = np.where(scores > t, 'Gentoo', 'Adelie')
    p = precision_score(y_test, pred, pos_label='Gentoo', zero_division=0)
    r = recall_score(y_test, pred, pos_label='Gentoo', zero_division=0)
    print(f'  s > {t:>4}       {p:.2f}        {r:.2f}')

### Read the figure and the table

The histogram shows the two species' scores: Adélies pile up on the left (negative), Gentoos on the right (positive), with a little overlap in the middle where mistakes happen. The dashed line is the threshold. Slide it and watch the table — at a low cutoff (s > −50) we catch every Gentoo (recall 1.00) but raise false alarms (precision 0.79); at a high cutoff (s > +50) every Gentoo call is correct (precision 1.00) but we miss many (recall 0.61). The default s > 0 sits between them (0.97, 0.94). One dial, the whole trade-off.

## The ROC curve

Rather than pick one threshold, we can show them all at once. For every possible cutoff, compute two rates: the **true-positive rate** (recall — the fraction of Gentoos caught) and the **false-positive rate** (the fraction of Adélies wrongly called Gentoo). Plotting the true-positive rate against the false-positive rate, as the threshold sweeps, traces the **ROC curve**.

In [ ]:
viz.plot_roc_curve(y_true, scores, label='bill only')
plt.show()
print(f'AUC (bill only): {roc_auc_score(y_true, scores):.3f}')

### Read the figure

Each point on the curve is one threshold. The bottom-left corner is a cutoff so high that nothing is called Gentoo (catch nothing, no false alarms); the top-right is a cutoff so low that everything is (catch all, all false alarms). A good classifier bows toward the **top-left** — high recall at a low false-alarm rate. The single-number summary is the **area under the curve (AUC)**: here **0.989**. AUC has a clean meaning — it is the probability that a randomly chosen Gentoo scores higher than a randomly chosen Adélie. The diagonal (AUC 0.5) is a coin flip.

## The precision-recall curve

ROC plots recall against the false-positive rate. When the positive class is *rare*, a second view is often more telling: the **precision-recall curve**, which plots precision against recall as the threshold sweeps. It answers "as we demand more recall, how much precision do we give up?"

In [ ]:
precisions, recalls, _ = precision_recall_curve(y_true, scores)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recalls, precisions, color=colors.COLORS['model'], linewidth=2)
ax.set_xlabel('recall')
ax.set_ylabel('precision')
ax.set_title('Precision-recall curve (bill only)')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0, 1.05)
plt.show()

### Read the figure

Reading from left to right, we can raise recall toward 1 (catch every Gentoo) while precision stays high, dropping only as we force the last few cases. A curve that stays high and far to the right is a strong classifier. Unlike ROC, the precision-recall curve does not flatter a model when the positive class is rare, which is why it is the one to watch when positives are scarce. (Our test set is near-balanced — 31 Gentoos to 38 Adélies — so here ROC and the precision-recall curve tell the same happy story; their disagreement shows up under heavy imbalance, which later chapters meet.)

## Fixed threshold vs swept threshold

Keep the distinction clear. **Accuracy, precision, recall** are read at *one* threshold — change the cutoff and they change. **ROC-AUC** is read across *all* thresholds at once: it measures how well the score *ranks* positives above negatives, regardless of where the cutoff eventually sits. A model can rank beautifully (high AUC) yet score poorly at a badly chosen threshold — ranking quality and operating point are separate questions.

In [ ]:
scores_2f = gentoo_score(X_train[datasets.PENGUINS_FEATURES], X_test[datasets.PENGUINS_FEATURES])

fig = viz.plot_roc_curve(y_true, scores, label='bill only', color=colors.COLORS['muted'])
viz.plot_roc_curve(
    y_true, scores_2f, label='both features', color=colors.COLORS['model'], ax=fig.axes[0]
)
plt.show()

print(f'AUC bill only     : {roc_auc_score(y_true, scores):.3f}')
print(f'AUC both features : {roc_auc_score(y_true, scores_2f):.3f}')

### Read the figure

The two-feature model (the one from notebook 05) is a near-perfect ranker: its ROC hugs the top-left corner and its AUC is **1.000**, against **0.989** for the bill-only model. The stronger model's curve sits above the weaker one's nearly everywhere — a higher curve, and a higher AUC, mean a better ranking of Gentoos above Adélies. This is what "a better classifier" looks like through the ROC lens.

## Your turn

1. You are building a screening pass that must not miss a single Gentoo, even at the cost of some false alarms. From the table in this notebook, which of the three thresholds would you choose, and what precision would you accept?
2. A classifier has AUC = 0.5. What does that say about how its scores rank the two species?
3. Explain how a model could have a high AUC and yet a mediocre accuracy at the threshold s > 0. (Hint: think about where the threshold sits relative to the scores.)

This closes the evaluation thread — notebook 09 turns from *measuring* a model to *the tension that shapes every model*: overfitting.

## What you built

You found the dial behind the label, and the tools to read it across all its settings:

- You turned the classifier's distances into a **score**, and saw the label was its sign.
- You watched the **threshold** trade precision against recall.
- You read a **ROC curve** and its **AUC**, and a **precision-recall curve**.
- You separated **threshold-fixed** metrics from the **threshold-swept** AUC.

**Vocabulary added:** score, decision threshold, true-positive rate, false-positive rate, ROC curve, AUC, precision-recall curve.

This closes the evaluation thread (notebooks 06–08): you can now judge a classifier honestly, from a single number to the full curve.

## References

- T. Fawcett (2006), *An introduction to ROC analysis*, Pattern Recognition Letters 27(8):861–874. DOI: 10.1016/j.patrec.2005.10.010
- A. Géron, *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd ed., O'Reilly, 2022 — chapter 3 (ROC, precision-recall, the threshold trade-off).
- scikit-learn developers, `sklearn.metrics` — `roc_curve`, `roc_auc_score`, `precision_recall_curve`.

---
Previous: **07 — The confusion matrix, precision & recall** · Next: **09 — Over-/under-fitting and the generalization gap**